In [ ]:
import sysfrom pathlib import Path# Handle multiple working directory scenarios# Case 1: Running from project root# Case 2: Running from notebooks/ directory# Case 3: Running as a Jupyter notebookcurrent_file = Path(__file__).resolve() if '__file__' in dir() else Path.cwd()if current_file.is_file():    # We're running a notebook file    notebook_dir = current_file.parent    project_root = notebook_dir.parentelse:    # We're in a directory    project_root = Path.cwd()    # If we're in notebooks/ subdirectory, go up one level    if project_root.name == 'notebooks':        project_root = project_root.parent# Add project root to path if not already thereif str(project_root) not in sys.path:    sys.path.insert(0, str(project_root))# Verify src module existssrc_path = project_root / 'src'if not src_path.exists():    raise RuntimeError(f"Could not find src/ directory. Project root: {project_root}")print(f"✓ Added {project_root} to sys.path")

# Understanding Scaled Dot-Product Attention

This notebook explains the core mechanism used by both GPT and masked-LM models: **scaled dot-product attention**.

We'll:
1. Visualize what attention does
2. Compare the NumPy reference vs PyTorch implementation
3. Show causal vs bidirectional masking
4. Demonstrate multi-head attention

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from src.common.sdpa_reference import softmax, scaled_dot_product_attention, multi_head_self_attention
from src.common.attention import MultiHeadSelfAttention

print("✓ Imports successful")

## Part 1: What is Scaled Dot-Product Attention?

The formula:
```
Attention(Q, K, V) = softmax((Q @ K^T) / sqrt(d_k)) @ V
```

Breaking it down:
1. **Q @ K^T** — Compute similarity between queries and keys
2. **/ sqrt(d_k)** — Scale to prevent saturation in softmax
3. **softmax()** — Convert similarities to probabilities
4. **@ V** — Weight values by attention probabilities

In [ ]:
# Create simple example
seq_len = 4
d_model = 8

# Generate random Q, K, V
Q = np.random.randn(seq_len, d_model)
K = np.random.randn(seq_len, d_model)
V = np.random.randn(seq_len, d_model)

print(f"Q shape: {Q.shape}")
print(f"K shape: {K.shape}")
print(f"V shape: {V.shape}")

# Compute attention
output, weights = scaled_dot_product_attention(Q, K, V, causal=False)

print(f"\nAttention output shape: {output.shape}")
print(f"Attention weights shape: {weights.shape}")
print(f"\nAttention weights (bidirectional):")
print(weights.round(3))

## Part 2: Causal vs Bidirectional Masking

In [ ]:
# Compare causal and bidirectional attention
output_bi, weights_bi = scaled_dot_product_attention(Q, K, V, causal=False)
output_causal, weights_causal = scaled_dot_product_attention(Q, K, V, causal=True)

print("BIDIRECTIONAL (no mask):")
print(weights_bi.round(3))
print(f"\nCAUSAL (lower triangular):")
print(weights_causal.round(3))

# Visualize the difference
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

im1 = ax1.imshow(weights_bi, cmap='viridis')
ax1.set_title('Bidirectional Attention\n(All positions can attend to all)')
ax1.set_xlabel('Key Position')
ax1.set_ylabel('Query Position')
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(weights_causal, cmap='viridis')
ax2.set_title('Causal Attention\n(Can only attend to past/present)')
ax2.set_xlabel('Key Position')
ax2.set_ylabel('Query Position')
plt.colorbar(im2, ax=ax2)

plt.tight_layout()
plt.show()

print("\nKey difference:")
print("- Bidirectional: Full matrix of weights")
print("- Causal: Lower triangular (upper triangle is 0)")

## Part 3: PyTorch Implementation vs NumPy Reference

In [ ]:
# Create PyTorch attention module
attn_torch = MultiHeadSelfAttention(d_model=8, n_heads=2, causal=False)

# Create tensor input
x_torch = torch.from_numpy(Q).unsqueeze(0).float()  # (1, seq, d_model)
output_torch = attn_torch(x_torch)

print(f"PyTorch attention output shape: {output_torch.shape}")
print(f"Expected: (batch=1, seq={seq_len}, d_model=8)")
print(f"\nPyTorch implementation successful!")

## Part 4: Multi-Head Attention

In [ ]:
# Multi-head attention splits d_model into n_heads
# Each head computes attention independently
# Then results are concatenated

d_model = 8
n_heads = 2
d_head = d_model // n_heads  # 4 per head

print(f"Model dimension: {d_model}")
print(f"Number of heads: {n_heads}")
print(f"Dimension per head: {d_head}")

# Create projection matrices for multi-head
x = np.random.randn(4, 8)  # (seq, d_model)
Wq = np.random.randn(8, 8)
Wk = np.random.randn(8, 8)
Wv = np.random.randn(8, 8)
Wo = np.random.randn(8, 8)

output_mh = multi_head_self_attention(x, Wq, Wk, Wv, Wo, n_heads=2, causal=False)

print(f"\nMulti-head attention output shape: {output_mh.shape}")
print(f"Same as input: {output_mh.shape == x.shape}")

print("\nHow it works:")
print("1. Split d_model into n_heads (8 → 2 heads × 4 dims)")
print("2. Compute attention for each head independently")
print("3. Concatenate results (4 + 4 = 8)")
print("4. Project back to d_model")

## Summary

Scaled dot-product attention is the core mechanism:

✓ **Simple formula** — Q @ K^T, scale, softmax, V projection  
✓ **Flexible masking** — Same code, different mask patterns  
✓ **Multi-head** — Parallel attention heads increase capacity  
✓ **Foundation** — Used by both GPT and masked-LM  

The only difference between autoregressive and bidirectional models is the attention mask pattern—the underlying math is identical.